## CHORD

This notebook analyzes CHORD training outputs. Run it from the CHORD repository root after training. Set `outdir` to the training output directory; the saved `config.yaml` supplies the input dataset and ortholog mapping paths.

In [ ]:
from pathlib import Path

import yaml
import scanpy as sc
import matplotlib.pyplot as plt

from chord.analysis import (
    evaluate_celltype,
    evaluate_batch,
    load_celltype_embeddings_from_h5ads,
    consensus_support_fuzzy_threshold,
    resolve_species_gene,
    plot_dendrogram_support_colors,
    build_label2expr_OG_adata,
    plot_tree_drift_fullcolor,
)
from chord.preprocessing import data_preprocess

## Loading CHORD results

In [ ]:
outdir = Path("results")

with (outdir / "config.yaml").open(encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

adata_cell = sc.read_h5ad(outdir / "adata_cell_embeddings.h5ad")
adata_gene = sc.read_h5ad(outdir / "adata_gene_embeddings.h5ad")
print(adata_cell)
print(adata_gene)

adata_cell_only = adata_cell[adata_cell.obs["item_type"] == "cell"].copy()
adata_celltype_only = adata_cell[adata_cell.obs["item_type"] == "celltype"].copy()
print(adata_cell_only)
print(adata_celltype_only)

## Visualizing the integration results

In [ ]:
sc.pp.neighbors(adata_cell_only)
sc.tl.umap(adata_cell_only)

In [ ]:
sc.pl.umap(
    adata_cell_only,
    color="cell_type",
    title="Cell type",
    show=False,
)

In [ ]:
sc.pl.umap(
    adata_cell_only,
    color="species",
    title="Species",
    show=False,
)

## Cell-type tree

The tree uses the saved cell-type embeddings. Branch support measures agreement across the input training runs; a single input gives 100% support. Add cell embedding files from independent runs to `embedding_paths` to compare branch agreement across runs.

In [ ]:
embedding_paths = [outdir / "adata_cell_embeddings.h5ad"]
embeddings, all_type_labels, row_labels = load_celltype_embeddings_from_h5ads(
    embedding_paths,
)
Z, support = consensus_support_fuzzy_threshold(
    embeddings,
    all_type_labels,
    row_labels_list=row_labels,
)
fig, ax = plot_dendrogram_support_colors(
    Z,
    support,
    all_type_labels,
    color_thr=[80, 50],
    orientation="right",
    flip_lr=True,
    size=[5, 9],
)

## Mapping homologous gene expression differences onto the cell-type tree

With CHORD, we can map homologous gene expression differences onto the cell-type tree to examine how these cross-species differences are distributed across cell-type relationships.

In [ ]:
homology_files = {
    (item["species1"], item["species2"]): Path(item["path"])
    for item in config["data"]["homology"]
}

species_all_adatas = {
    item["name"]: (
        data_preprocess(
            item["h5ad"],
            cpm=config["data"].get("cpm", 10000),
            is_preprocessed=item.get("is_preprocessed", False),
            select_hvg=False,
        ),
        item["label_col"],
    )
    for item in config["data"]["species"]
}

gene_name = "foxa"

In [ ]:
species_gene = resolve_species_gene(homology_files, species_all_adatas, "frog", gene_name)
label2expr = build_label2expr_OG_adata(
    all_type_labels,
    species_all_adatas,
    species_gene
)
fig, ax = plot_tree_drift_fullcolor(
    Z, all_type_labels, label2expr,size=(5, 9),
    cmap_name="bwr",flip_lr=True
)
plt.show()

## Evaluation integration

In [ ]:
evaluate_batch(adata_cell_only)
evaluate_celltype(adata_cell_only)

## Visualizing gene embeddings

In [ ]:
sc.pp.neighbors(adata_gene)
sc.tl.umap(adata_gene)
sc.pl.umap(
    adata_gene,
    color="species",
    title="species",
    show=False,
)
